In [1]:
from transformers import AutoTokenizer
from bertviz.transformers_neuron_view import BertModel
from bertviz.neuron_view import show

In [2]:
model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = BertModel.from_pretrained(model_ckpt)
text = "time flies like an arrow"
show(model,"bert",tokenizer,text,display_mode="light",layer=0,head=8)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
inputs = tokenizer(text,return_tensors='pt',add_special_tokens=False)
inputs.input_ids

tensor([[ 2051, 10029,  2066,  2019,  8612]])

In [4]:
from torch import nn
from transformers import AutoConfig

config = AutoConfig.from_pretrained(model_ckpt)
token_emb = nn.Embedding(config.vocab_size,config.hidden_size)
token_emb

/Users/anmolagrawal/opt/anaconda3/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embedding(30522, 768)

In [5]:
input_embeds = token_emb(inputs.input_ids)
input_embeds.size()

torch.Size([1, 5, 768])

In [6]:
import torch
from math import sqrt

query = key = value = input_embeds
dim_k = query.size(-1)
scores = torch.bmm(query, key.transpose(1,2))/sqrt(dim_k)
scores.size()

torch.Size([1, 5, 5])

In [7]:
import torch.nn.functional as F

weights = F.softmax(scores,dim=-1)
print(weights.sum(dim=-1))
print(weights.shape)

tensor([[1., 1., 1., 1., 1.]], grad_fn=<SumBackward1>)
torch.Size([1, 5, 5])


In [8]:
attn_outputs = torch.bmm(weights, value)
attn_outputs.shape

torch.Size([1, 5, 768])

In [20]:
def scaled_dot_product_attention(query,key,value,mask=None):
    dim_k = query.size(-1)
    scores = torch.bmm(query, key.transpose(1,2))/sqrt(dim_k)
    if mask is not None:
        scores.masked_fill(mask==0,float("-inf"))
    weights = F.softmax(scores,dim=-1)
    return torch.bmm(weights, value)

In [10]:
class AttentionHead(nn.Module):
    def __init__(self, embed_dim,head_dim) -> None:
        super().__init__()
        self.q = nn.Linear(embed_dim,head_dim)
        self.k = nn.Linear(embed_dim,head_dim)
        self.v = nn.Linear(embed_dim,head_dim)

    def forward(self,hidden_sate):
        attn_outputs = scaled_dot_product_attention(self.q(hidden_sate), self.k(hidden_sate), self.v(hidden_sate))
        return attn_outputs

In [11]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config) -> None:
        super().__init__()
        embed_dim = config.hidden_size
        num_heads = config.num_attention_heads
        head_dim = embed_dim//num_heads
        self.heads = nn.ModuleList([AttentionHead(embed_dim,head_dim) for _ in range(num_heads)])
        self.output_linear = nn.Linear(embed_dim,embed_dim)

    def forward(self, hidden_sate):
        x = torch.cat([h(hidden_sate) for h in self.heads],dim=-1)
        x = self.output_linear(x)
        return x

In [12]:
multihead_attn = MultiHeadAttention(config)
attn_output = multihead_attn(input_embeds)
attn_output.size()

torch.Size([1, 5, 768])

In [13]:
class FeedForward(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.linear_1 = nn.Linear(config.hidden_size,config.intermediate_size)
        self.linear_2 = nn.Linear(config.intermediate_size,config.hidden_size)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self,x):
        x = self.linear_1(x)
        x = self.gelu(x)
        x = self.linear_2(x)
        x = self.dropout(x)
        return x


In [14]:
feedforwad = FeedForward(config)
ff_output = feedforwad(attn_output)
ff_output.size()

torch.Size([1, 5, 768])

In [15]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.layer_norm_1 = nn.LayerNorm(config.hidden_size)
        self.layer_norm_2 = nn.LayerNorm(config.hidden_size)
        self.attention = MultiHeadAttention(config)
        self.feed_forward = FeedForward(config)
    
    def forward(self,x):
        hidden_state = self.layer_norm_1(x)
        x = x + self.attention(hidden_state)
        x = x + self.feed_forward(self.layer_norm_2(x))
        return x

In [16]:
encoder_layer = TransformerEncoderLayer(config)
encoder_layer(input_embeds).size()

torch.Size([1, 5, 768])

In [17]:
class Embedding(nn.Module):
    def __init__(self,config) -> None:
        super().__init__()
        self.token_embeddings = nn.Embedding(config.vocab_size,config.hidden_size)
        self.position_embeddings = nn.Embedding(config.max_position_embeddings,config.hidden_size)
        self.layer_norm = nn.LayerNorm(config.hidden_size,eps=1e-12)
        self.dropout = nn.Dropout()
    
    def forward(self, input_ids):
        seq_length = input_ids.size(1)
        position_ids = torch.arange(seq_length,dtype=torch.long).unsqueeze(0)
        token_embeddings = self.token_embeddings(input_ids)
        position_embeddings = self.position_embeddings(position_ids)
        embeddings = token_embeddings + position_embeddings
        embeddings = self.layer_norm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings

In [18]:
class TransformerEncoder(nn.Module):
    def __init__(self, config) -> None:
        super().__init__()
        self.embeddings = Embedding(config)
        self.layers = nn.ModuleList([TransformerEncoderLayer(config) for _ in range(config.num_hidden_layers)])
    
    def forward(self,x):
        x = self.embeddings(x)
        for layer in self.layers:
            x = layer(x)
        return x

In [19]:
encoder = TransformerEncoder(config)
encoder(inputs.input_ids).size()

torch.Size([1, 5, 768])